# Analysis of the sample of 25 configurations

In [1]:
import pandas as pd

In [2]:
best_config = None

In [3]:
data = pd.read_json("results/eval_results.json")

data

,name,avg_distance,avg_retrieval_time,avg_generation_time,faithfulness,answer_relevancy,context_precision
0,e5-base-v2 + gpt-oss-120b | cs=1000 ov=100 n=3...,0.275208,0.035,4.779,0.854,0.860,0.710
1,all-MiniLM-L6-v2 + gpt-oss-120b | cs=500 ov=20...,0.684608,0.011,7.368,0.734,0.718,0.300
2,all-MiniLM-L6-v2 + flan-t5-base | cs=2000 ov=2...,0.684608,0.008,0.887,0.493,0.346,0.366
3,bge-small-en-v1.5 + flan-t5-base | cs=1000 ov=...,0.415292,0.019,2.254,0.739,0.704,0.702
4,all-mpnet-base-v2 + gpt-oss-120b | cs=1000 ov=...,0.793863,0.032,3.604,0.616,0.645,0.291
5,all-mpnet-base-v2 + flan-t5-base | cs=1000 ov=...,0.793863,0.032,2.372,0.523,0.381,0.260
6,e5-base-v2 + flan-t5-base | cs=1500 ov=200 n=5...,0.275208,0.032,2.912,0.843,0.804,0.708
7,e5-base-v2 + llama-3.1-8b-instant | cs=1000 ov...,0.275208,0.032,5.214,0.844,0.852,0.710
8,bge-base-en-v1.5 + flan-t5-large | cs=500 ov=1...,0.420329,0.047,1.715,0.673,0.618,0.671
9,all-MiniLM-L6-v2 + flan-t5-large | cs=500 ov=2...,0.684608,0.013,2.404,0.393,0.280,0.366


In [4]:
dist_min = data.avg_distance.min()

print("Minimal distance for retrieval:", dist_min)

Minimal distance for retrieval: 0.275208321809768


In [5]:
best_encoders = data[data.avg_distance == dist_min].name
best_encoders = best_encoders.apply(lambda x: x.split("+")[0])

best_encoders

0     e5-base-v2 
6     e5-base-v2 
7     e5-base-v2 
12    e5-base-v2 
14    e5-base-v2 
16    e5-base-v2 
17    e5-base-v2 
Name: name, dtype: str

As we can see, the best encoder (by average distance in retrieval) is e5-base-v2, but let's firstly confirm the time to retrieve is short enough:

In [6]:
RT_BE = data[data.name.str.contains(best_encoders[0])]
AVG_RT_BE = RT_BE.avg_retrieval_time.mean()

print(f"AVG retrieval time for {best_encoders[0]}is {AVG_RT_BE}")

AVG retrieval time for e5-base-v2 is 0.03214285714285715


I suppose the retrieval time of 0.03s is acceptable for a CLI-like program processing user requests. Let's check what LLM's did their best:

In [7]:
best_llms = data[(data.faithfulness > 0.8) & (data.answer_relevancy > 0.8) & 
                                  (data.context_precision > 0.7)]
best_llms.reset_index(drop=True, inplace=True)

best_llms

,name,avg_distance,avg_retrieval_time,avg_generation_time,faithfulness,answer_relevancy,context_precision
0,e5-base-v2 + gpt-oss-120b | cs=1000 ov=100 n=3...,0.275208,0.035,4.779,0.854,0.860,0.710
1,e5-base-v2 + flan-t5-base | cs=1500 ov=200 n=5...,0.275208,0.032,2.912,0.843,0.804,0.708
2,e5-base-v2 + llama-3.1-8b-instant | cs=1000 ov...,0.275208,0.032,5.214,0.844,0.852,0.710
3,e5-base-v2 + llama-3.1-8b-instant | cs=2000 ov...,0.275208,0.033,8.556,0.858,0.856,0.708
4,e5-base-v2 + gpt-oss-120b | cs=1500 ov=200 n=3...,0.275208,0.031,4.663,0.853,0.846,0.710
5,e5-base-v2 + flan-t5-base | cs=1500 ov=200 n=5...,0.275208,0.031,2.917,0.843,0.804,0.708


There are enough configs that could be used in production pipeline. Now we check them in a more detailed way:

In [8]:
print(*best_llms.name, sep="\n")

e5-base-v2 + gpt-oss-120b | cs=1000 ov=100 n=3 th=0.8
e5-base-v2 + flan-t5-base | cs=1500 ov=200 n=5 th=1
e5-base-v2 + llama-3.1-8b-instant | cs=1000 ov=200 n=3 th=1
e5-base-v2 + llama-3.1-8b-instant | cs=2000 ov=100 n=5 th=1.1
e5-base-v2 + gpt-oss-120b | cs=1500 ov=200 n=3 th=1
e5-base-v2 + flan-t5-base | cs=1500 ov=200 n=5 th=0.8


In [9]:
(best_llms.faithfulness + best_llms.answer_relevancy +
                best_llms.context_precision).nlargest(3)

0    2.424
3    2.422
4    2.409
dtype: float64

Configurations 0 and 3 have the best results in terms of faithfulness, answer relevancy, and context precision. At the same time, there are no obvious drawbacks in any of them. I study the two settings:

In [10]:
best_llms.iloc[0, :]

name                   e5-base-v2 + gpt-oss-120b | cs=1000 ov=100 n=3...
avg_distance                                                    0.275208
avg_retrieval_time                                                 0.035
avg_generation_time                                                4.779
faithfulness                                                       0.854
answer_relevancy                                                    0.86
context_precision                                                   0.71
Name: 0, dtype: object

In [11]:
best_llms.iloc[3, :]

name                   e5-base-v2 + llama-3.1-8b-instant | cs=2000 ov...
avg_distance                                                    0.275208
avg_retrieval_time                                                 0.033
avg_generation_time                                                8.556
faithfulness                                                       0.858
answer_relevancy                                                   0.856
context_precision                                                  0.708
Name: 3, dtype: object

We can notice that the average generation time for the first chosen configuration is `44% lower` than for the second one. This drastically impacts UX, so we are proceeding with the first configuration.

In [12]:
best_config = best_llms.iloc[0, :]
best_config = best_config.reset_index()

The final look of the configuration:

In [13]:
print("="*68)
print(f"BEST CONFIG IS {best_config.iloc[0, 1]}")
print("="*68)
best_config = best_config.iloc[1:, :]
best_config = best_config.to_string(index=False, header=False).split("\n")
best_config = [i.strip() for i in best_config]
print("PERFORMANCE:")
for line in best_config:
    key, value = line.split()
    print(f"\t{key:<25} {value:>8}")
print("="*68)

BEST CONFIG IS e5-base-v2 + gpt-oss-120b | cs=1000 ov=100 n=3 th=0.8
PERFORMANCE:
	avg_distance              0.275208
	avg_retrieval_time           0.035
	avg_generation_time          4.779
	faithfulness                 0.854
	answer_relevancy              0.86
	context_precision             0.71


For the encoder we will use e5-base-v2 along with gpt-oss-120b from Groq API as LLM.
Regarding the overlap and chunk size, they will remain unchanged.
However, I will experiment manually through in-deployment interface with the correct distance threshold (for now th=0.8). It is used to limit LLM's answers:
``` python
        best_distance = results['distances'][0][0]
        if best_distance > self.distance_threshold:
            return "This question doesn't seem to be about Blender."
```
The reason for that is, LLM's are trained to be very helpful, so sometimes they make up information or can answer questions irrelevant to the RAG (for example, "code me a bubble sort algorithm in python"). This cannot be changed through prompt structure or role settings, as the end-user can anyways surpass restrictions with a good prompting strategy. Therefore, I decided to limit these capabilities in a more deterministic way to prevent undesired behavior. Also, this could be a drawback if, for example, in my case LLM can be trained on one documentation version, but the RAG uses another one, so sometimes things that are non-existing in the documentation are completely ignored by the model, which gives out information it was trained on (or, even worse, hallucination) rather than information present in the database, poorly influencing the whole purpose of RAG.

Additional information: n=3 is number of retrieved results 3, meaning the amount of information retrieved from DB for each query. We will proceed with 3 in production.